# Bayesian KAN — 1D Toy Validation

Validates the B-coef Bayesian KAN on a 1D sinusoidal regression problem
with a deliberate gap in the training data.  
The gap region tests whether the model correctly reports **high epistemic uncertainty**
where no training data exist, while keeping uncertainty low in the observed regions.

**Comparisons**
- `BayesianKAN` (B-coef): Gaussian posteriors over B-spline coefficients (this work)
- `BayesianNN` (Bayes-by-Backprop): Gaussian posteriors over MLP weights (Tyler's code)

**Dataset** — Case 6 (sinusoidal with gap):  
Training in $x \in [-0.2, 0.2] \cup [0.6, 1.0]$, gap in $[0.2, 0.6]$, test on $[-0.5, 1.5]$.  
True function: $f(x) = x + 0.3\sin(2\pi x) + 0.3\sin(4\pi x)$, noise $\sigma = 0.02$.


## 1. Setup

In [ ]:
import sys
import os

# Add the repository root to the path so src/ is importable.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from bkan.models import BayesianKAN, BayesianNN
from bkan.training.trainer import BNNTrainer, TrainingConfig
from bkan.data.toy_problems import generate_case6_sinusoidal

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## 2. Dataset

In [ ]:
x_train, y_train, x_test, _ = generate_case6_sinusoidal(
    n_train=1000, n_test=1000, seed=SEED
)

# True noiseless function for reference
x_np = x_test.numpy().flatten()
y_true = x_np + 0.3 * np.sin(2 * np.pi * x_np) + 0.3 * np.sin(4 * np.pi * x_np)

print(f'Training points : {len(x_train)}')
print(f'Test points     : {len(x_test)}')
print(f'x_train range   : [{x_train.min():.2f}, {x_train.max():.2f}]')

fig, ax = plt.subplots(figsize=(10, 3))
ax.scatter(x_train.numpy(), y_train.numpy(), s=4, alpha=0.4,
           color='steelblue', label='Training data')
ax.plot(x_np, y_true, 'k-', lw=1.5, label='True function')
ax.axvspan(0.2, 0.6, alpha=0.12, color='orange', label='Gap (no training data)')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Dataset — Sinusoidal with gap')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 3. Training configuration

Both models use `BNNTrainer` and `TrainingConfig`, but with **separate** schedules.  
BayesianKAN converges with short KL annealing (500 epochs).  
BayesianNN (BBB) requires longer annealing (2000 epochs): applying the full KL weight
at epoch 500 forces MLP weights toward the prior before the network has converged,
destroying the learned fit.


In [ ]:
EPOCHS        = 5000
BATCH_SIZE    = 256
LEARNING_RATE = 1e-3
N_MC_TRAIN    = 1      # MC samples per ELBO estimate during training
N_MC_PREDICT  = 200    # MC samples for predictive distribution at test time

# Short annealing suffices for BayesianKAN: the spline parameterisation
# converges faster than a dense MLP.
config_bkan = TrainingConfig(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    kl_annealing_epochs=500,
    n_mc_samples=N_MC_TRAIN,
    early_stopping_patience=500,
    scheduler='cosine',
    verbose=True,
    print_every=500,
)

# Longer annealing for BayesianNN: prevents KL from collapsing weights
# before the loss landscape is well-explored.
config_bnn = TrainingConfig(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    kl_annealing_epochs=2000,
    n_mc_samples=N_MC_TRAIN,
    early_stopping_patience=500,
    scheduler='cosine',
    verbose=True,
    print_every=500,
)


## 4. Train BayesianKAN

In [ ]:
torch.manual_seed(SEED)

model_bkan = BayesianKAN(
    input_dim=1,
    hidden_dims=[8, 8],
    output_dim=1,
    num=5,            # G = 5 grid intervals per spline
    k=3,              # cubic B-splines
    prior_std=1.0,
    learn_noise=True,
    coef_log_var_init=-5.0,
    grid_range=[-0.6, 1.1],  # covers the test domain with margin
    device=DEVICE,
)

total_params = sum(p.numel() for p in model_bkan.parameters() if p.requires_grad)
print(f'BayesianKAN trainable parameters: {total_params:,}')

trainer_bkan = BNNTrainer(model=model_bkan, config=config_bkan, device=DEVICE)
history_bkan = trainer_bkan.train(x_train, y_train)

## 5. Train BayesianNN (baseline)

In [ ]:
torch.manual_seed(SEED)

model_bnn = BayesianNN(
    input_dim=1,
    hidden_dims=[50, 50],
    output_dim=1,
    prior_std=1.0,
    learn_noise=True,
    activation='tanh',
)

total_params_bnn = sum(p.numel() for p in model_bnn.parameters() if p.requires_grad)
print(f'BayesianNN trainable parameters: {total_params_bnn:,}')

trainer_bnn = BNNTrainer(model=model_bnn, config=config_bnn, device=DEVICE)
history_bnn = trainer_bnn.train(x_train, y_train)

## 6. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, history, label, color in [
    (axes[0], history_bkan, 'BayesianKAN', 'C0'),
    (axes[1], history_bnn,  'BayesianNN',  'C1'),
]:
    epochs_arr = np.arange(1, len(history.loss) + 1)
    ax.semilogy(epochs_arr, history.nll, color=color, lw=1.5, label='NLL')
    ax.semilogy(epochs_arr, history.kl,  color=color, lw=1.5, ls='--', alpha=0.6, label='KL / batch')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss component (log scale)')
    ax.set_title(f'{label} — training loss')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Predictive distributions

In [ ]:
x_test_dev = x_test.to(DEVICE)

# BayesianKAN
mean_bkan, std_bkan, _ = model_bkan.predict(x_test_dev, n_samples=N_MC_PREDICT)
_, ep_std_bkan, al_std_bkan = model_bkan.predict_decomposed(x_test_dev, n_samples=N_MC_PREDICT)

mean_bkan    = mean_bkan.cpu().numpy().flatten()
std_bkan     = std_bkan.cpu().numpy().flatten()
ep_std_bkan  = ep_std_bkan.cpu().numpy().flatten()
al_std_bkan  = al_std_bkan.cpu().numpy().flatten()

# BayesianNN
mean_bnn, std_bnn, _ = model_bnn.predict(x_test.to(DEVICE), n_samples=N_MC_PREDICT)
_, ep_std_bnn, al_std_bnn = model_bnn.predict_decomposed(x_test.to(DEVICE), n_samples=N_MC_PREDICT)

mean_bnn    = mean_bnn.cpu().numpy().flatten()
std_bnn     = std_bnn.cpu().numpy().flatten()
ep_std_bnn  = ep_std_bnn.cpu().numpy().flatten()
al_std_bnn  = al_std_bnn.cpu().numpy().flatten()

print('BayesianKAN  — mean epistemic std in gap [0.2,0.6]:',
      ep_std_bkan[(x_np > 0.2) & (x_np < 0.6)].mean().round(4))
print('BayesianNN   — mean epistemic std in gap [0.2,0.6]:',
      ep_std_bnn[(x_np > 0.2) & (x_np < 0.6)].mean().round(4))
print('BayesianKAN  — mean epistemic std in training region:',
      ep_std_bkan[(x_np < 0.2) | (x_np > 0.6)].mean().round(4))
print('BayesianNN   — mean epistemic std in training region:',
      ep_std_bnn[(x_np < 0.2) | (x_np > 0.6)].mean().round(4))

## 8. Visualisation — total uncertainty

In [ ]:
def _plot_prediction(ax, x, y_true, x_train, y_train,
                     mean, std, title, color):
    """Plot predictive mean ± 2σ against true function and training data."""
    ax.fill_between(x, mean - 2*std, mean + 2*std,
                    alpha=0.25, color=color, label=r'Mean $\pm\,2\sigma_{\rm total}$')
    ax.plot(x, mean, color=color, lw=2.0, label='Predictive mean')
    ax.plot(x, y_true, 'k--', lw=1.2, alpha=0.7, label='True function')
    ax.scatter(x_train, y_train, s=4, alpha=0.35, color='k', zorder=5,
               label='Training data')
    ax.axvspan(0.2, 0.6, alpha=0.08, color='orange')
    ax.set_xlim(x.min(), x.max())
    ax.set_xlabel('x', fontsize=11)
    ax.set_ylabel('y', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=8, loc='upper left')
    # Clip y-axis to the function range so boundary extrapolation spikes
    # (which correctly signal high uncertainty) do not collapse the scale.
    y_margin = 0.8 * (y_true.max() - y_true.min())
    ax.set_ylim(y_true.min() - y_margin, y_true.max() + y_margin)
    ax.grid(True, alpha=0.25)


fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

_plot_prediction(
    axes[0], x_np, y_true,
    x_train.numpy().flatten(), y_train.numpy().flatten(),
    mean_bkan, std_bkan,
    title='BayesianKAN (B-coef, [1,8,8,1])',
    color='C0',
)
_plot_prediction(
    axes[1], x_np, y_true,
    x_train.numpy().flatten(), y_train.numpy().flatten(),
    mean_bnn, std_bnn,
    title='BayesianNN (BBB, [1,50,50,1])',
    color='C1',
)

# Annotate gap
for ax in axes:
    ax.text(0.4, ax.get_ylim()[0] + 0.05 * (ax.get_ylim()[1] - ax.get_ylim()[0]),
            'gap', ha='center', fontsize=9, color='darkorange', style='italic')

fig.suptitle('Predictive distributions — 1D sinusoidal with gap', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('bkan_1d_total_uncertainty.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: bkan_1d_total_uncertainty.pdf')

## 9. Visualisation — epistemic vs aleatoric decomposition

In [ ]:
def _plot_decomposed(ax, x, mean, ep_std, al_std, title, color_ep, color_al):
    """Plot epistemic and aleatoric uncertainty bands separately."""
    # Epistemic band: mean ± 2 * epistemic std
    ax.fill_between(x, mean - 2*ep_std, mean + 2*ep_std,
                    alpha=0.30, color=color_ep,
                    label=r'$\pm\,2\sigma_{\rm epistemic}$')
    # Aleatoric band (centred on mean, inner band)
    ax.fill_between(x, mean - 2*al_std, mean + 2*al_std,
                    alpha=0.40, color=color_al,
                    label=r'$\pm\,2\sigma_{\rm aleatoric}$')
    ax.plot(x, mean, color=color_ep, lw=2.0, label='Predictive mean')
    ax.axvspan(0.2, 0.6, alpha=0.08, color='orange')
    ax.set_xlim(x.min(), x.max())
    ax.set_xlabel('x', fontsize=11)
    ax.set_ylabel('y', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.25)
    # Clip to data range; boundary extrapolation spikes are real but
    # should not dominate the visualisation.
    ax.set_ylim(-2.0, 3.0)


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

_plot_decomposed(
    axes[0], x_np, mean_bkan, ep_std_bkan, al_std_bkan,
    title='BayesianKAN — uncertainty decomposition',
    color_ep='C0', color_al='C2',
)
_plot_decomposed(
    axes[1], x_np, mean_bnn, ep_std_bnn, al_std_bnn,
    title='BayesianNN — uncertainty decomposition',
    color_ep='C1', color_al='C2',
)

fig.suptitle('Epistemic vs aleatoric uncertainty — 1D sinusoidal with gap',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('bkan_1d_decomposed_uncertainty.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: bkan_1d_decomposed_uncertainty.pdf')

## 10. Uncertainty profile along x

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

for ax, ep_std, al_std, label, color in [
    (axes[0], ep_std_bkan, al_std_bkan, 'BayesianKAN', 'C0'),
    (axes[1], ep_std_bnn,  al_std_bnn,  'BayesianNN',  'C1'),
]:
    ax.plot(x_np, ep_std, color=color,  lw=1.8, label='Epistemic std')
    ax.plot(x_np, al_std, color='C2',   lw=1.8, ls='--', label='Aleatoric std')
    ax.axvspan(0.2, 0.6, alpha=0.10, color='orange', label='Gap')
    ax.axvline(0.2, color='orange', lw=0.8, ls=':')
    ax.axvline(0.6, color='orange', lw=0.8, ls=':')
    ax.set_ylabel('Std', fontsize=10)
    ax.set_title(label, fontsize=11)
    ax.legend(fontsize=8, ncol=3, loc='upper right')
    ax.grid(True, alpha=0.25)

axes[1].set_xlabel('x', fontsize=11)
fig.suptitle('Uncertainty profile along x', fontsize=13)
plt.tight_layout()
plt.savefig('bkan_1d_uncertainty_profile.pdf', bbox_inches='tight')
plt.show()

## 11. Quantitative metrics

Evaluate on the test set using standard metrics from the BNN benchmark literature
(Hernandez-Lobato & Adams 2015):  
- **RMSE**: root-mean-square error on the mean prediction  
- **NLL**: test negative log-likelihood under the Gaussian predictive (lower is better)  
- **Coverage 95%**: fraction of test points inside the 95% predictive interval (should be ≈ 0.95)

In [ ]:
def compute_metrics(mean, std, y_true_np, label):
    """RMSE, test NLL, and 95% coverage for a Gaussian predictive."""
    rmse = np.sqrt(np.mean((mean - y_true_np) ** 2))

    # Gaussian NLL: 0.5 * [log(2 pi sigma^2) + (y - mu)^2 / sigma^2]
    nll = 0.5 * np.mean(
        np.log(2 * np.pi * std**2 + 1e-8)
        + (mean - y_true_np) ** 2 / (std**2 + 1e-8)
    )

    lower = mean - 1.96 * std
    upper = mean + 1.96 * std
    coverage = np.mean((y_true_np >= lower) & (y_true_np <= upper))

    print(f'{label}:')
    print(f'  RMSE     = {rmse:.4f}')
    print(f'  Test NLL = {nll:.4f}')
    print(f'  95% cov  = {coverage:.3f}  (ideal: 0.950)')
    print()
    return {'rmse': rmse, 'nll': nll, 'coverage': coverage}


metrics_bkan = compute_metrics(mean_bkan, std_bkan, y_true, 'BayesianKAN')
metrics_bnn  = compute_metrics(mean_bnn,  std_bnn,  y_true, 'BayesianNN')

## 12. Metrics in the gap vs training region

In [ ]:
gap_mask    = (x_np >= 0.2) & (x_np <= 0.6)
train_mask  = ~gap_mask

print('Mean epistemic std')
print(f"  BayesianKAN — gap    : {ep_std_bkan[gap_mask].mean():.4f}")
print(f"  BayesianKAN — trained: {ep_std_bkan[train_mask].mean():.4f}")
print(f"  Ratio (gap/trained)  : {ep_std_bkan[gap_mask].mean() / ep_std_bkan[train_mask].mean():.2f}x")
print()
print(f"  BayesianNN  — gap    : {ep_std_bnn[gap_mask].mean():.4f}")
print(f"  BayesianNN  — trained: {ep_std_bnn[train_mask].mean():.4f}")
print(f"  Ratio (gap/trained)  : {ep_std_bnn[gap_mask].mean() / ep_std_bnn[train_mask].mean():.2f}x")